In [ ]:
import os

# Set this environment variable BEFORE importing numpy/sklearn if possible,
# or at least before running the compute-heavy task.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"  # Often helpful to restrict OMP as well


import scanpy as sc
import numpy as np
import pandas as pd
import plotnine as gg
from data_resources import load_crispri_data, load_fitness_data

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Extract the tab10 colors as hex codes
tab10_colors = [mcolors.to_hex(plt.get_cmap("tab10")(i)) for i in range(10)]

In [ ]:
timeseries_df = pd.read_pickle(
    "/workspace/data/Eaton_2025/Data/lDE20_Imaging/Clustering/2023-01-23_sgRNA_Timeseries_df.pkl"
)
timeseries_df.info()

In [ ]:
# Protocol
# 1. transcripts: visualization in low-dimensional space
# 2. transcripts: visualization per guide
# 2. fitness: boxplots
# 3. timeseries: boxplots

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
fitness_df = load_fitness_data()

# adata.obs = adata.obs.merge(fitness_df, how="left", left_on="spacer", right_index=True)

In [ ]:
adata

In [ ]:
# from scvi.model import SCVI

# SCVI.setup_anndata(adata, batch_key="rt_bc", layer="reads")
# model = SCVI(adata)
# model.train()

# adata.obsm["X_scVI"] = model.get_latent_representation()
# sc.pp.neighbors(adata, use_rep="X_scVI")
# sc.tl.umap(adata)

In [ ]:
# adata.write_h5ad(
#     "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
# )

In [ ]:
adata.obs["umap_x"] = adata.obsm["X_umap"][:, 0]
adata.obs["umap_y"] = adata.obsm["X_umap"][:, 1]

In [ ]:
genes_in_path = ["lpxA", "lpxC", "lpxD", "lpxB", "lpxK"]
adata_sub = adata[adata.obs["gene"].isin(genes_in_path)].copy()
adata_sub.obs["gene"] = adata_sub.obs["gene"].astype(str)
adata_sub.obs["gene"] = pd.Categorical(
    adata_sub.obs["gene"], categories=genes_in_path, ordered=True
)

In [ ]:
(
    gg.ggplot(
        adata.obs,
        gg.aes(x="umap_x", y="umap_y"),
    )
    + gg.geom_point()
    + gg.geom_point(adata_sub.obs, gg.aes(x="umap_x", y="umap_y", color="gene"))
)

In [ ]:
def vis_spacers(adata, gene, color="spacer"):
    adata_sub_onegene = adata[adata.obs["gene"] == gene].copy()

    return (
        gg.ggplot(
            adata.obs,
            gg.aes(x="umap_x", y="umap_y"),
        )
        + gg.geom_point(size=0.5, stroke=0)
        + gg.geom_point(adata_sub_onegene.obs, gg.aes(x="umap_x", y="umap_y", color=color))
    )

In [ ]:
gene = "lpxK"
fig1 = vis_spacers(adata, gene) + gg.labs(title=gene) + gg.theme(figure_size=(6, 3))
display(fig1)
fig2 = vis_spacers(adata, gene, color="T4") + gg.labs(title=gene) + gg.theme(figure_size=(6, 3))
display(fig2)

In [ ]:
gene = "lpxA"
fig1 = vis_spacers(adata, gene) + gg.labs(title=gene) + gg.theme(figure_size=(6, 3))
display(fig1)
fig2 = vis_spacers(adata, gene, color="T4") + gg.labs(title=gene) + gg.theme(figure_size=(6, 3))
display(fig2)

In [ ]:
gene = "lpxB"
fig1 = vis_spacers(adata, gene) + gg.labs(title=gene) + gg.theme(figure_size=(6, 3))
display(fig1)
fig2 = vis_spacers(adata, gene, color="T4") + gg.labs(title=gene) + gg.theme(figure_size=(6, 3))
display(fig2)

In [ ]:
group1 = ["lpxA", "lpxD", "lpxC"]
group2 = ["lpxB", "lpxK"]
X1 = adata[adata.obs["gene"].isin(group1).values, :].layers["cp10k"].toarray()
X2 = adata[adata.obs["gene"].isin(group2).values, :].layers["cp10k"].toarray()
X1.shape, X2.shape

In [ ]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

results = ttest_ind(X1, X2, axis=0, equal_var=False)
pvals = results.pvalue
pvals[np.isnan(pvals)] = 1.0
padjs = multipletests(pvals, method="fdr_bh")[1]

effect_sizes = results.statistic
effect_sizes[np.isnan(effect_sizes)] = 0.0

de_results = pd.DataFrame(
    {
        "padj": padjs,
        "effect_size": effect_sizes,
        "gene": adata.var_names,
    }
)
de_results

In [ ]:
de_results[de_results["padj"] < 0.05]

In [ ]:
gene_names = de_results.sort_values(by="padj", ascending=True).head(50).values
for gene in gene_names:
    print(gene)

In [ ]:
de_genes = [
    "wza",
    "wzc",
    "ivy",
    "osmB",
    "entC",
    "arnB",
    "arnC",
    "yigZ",
    "eptC",
    "rpoS",
    "katE",
    "dps",
    "hdeA",
    "hdeB",
    "hdeD",
    "gadC",
    "amyA",
]

In [ ]:
adata_sub = adata[adata.obs["gene"].isin(group1 + group2).values, :]
sc.pl.dotplot(adata_sub, de_genes, groupby="gene", layer="cp10k", categories_order=group1 + group2)

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
df = timeseries_df.loc[lambda x: x["Gene"].isin(group1 + group2)].loc[
    lambda x: x["N Mismatch"] == 0
]
df.info()

[5 tools called]

The provided dataframe contains processed phenotypic data from the MARLIN platform. Based on the paper and dataset structure, each row corresponds to a specific **sgRNA** (genetic perturbation targeting an essential gene). The "Kernel Trace" columns contain **time-series data** representing how the population phenotype evolves over the course of the experiment (typically following CRISPRi induction).

Here is the description of the columns:

### **Phenotypic Time-Series (Kernel Traces)**
These columns contain 1D arrays (time-series) representing the population-averaged phenotype at each timepoint of the experiment (e.g., across generations or hours post-induction).

*   **0-8: Kernel Trace: Birth/Division/Delta: Length/Width/Volume**
    *   *Description*: Arrays containing the average physical dimensions of cells at specific cell-cycle stages over the experiment time.
        *   `Birth`: Dimensions at the start of the cell cycle.
        *   `Division`: Dimensions at the end of the cell cycle.
        *   `Delta`: The change in dimension during the cell cycle (e.g., Elongation = Division Length - Birth Length).
    *   *Confidence*: High

*   **9-10: Kernel Trace: Final/Delta Timepoints**
    *   *Description*: Arrays containing the average duration of the cell cycle measured in imaging frames. `Final` likely refers to the cycle end, and `Delta` to the duration.
    *   *Confidence*: Medium

*   **11-12: Kernel Trace: Final time (s) / Delta time (s)**
    *   *Description*: Arrays containing the average cell cycle duration in seconds. `Delta time (s)` corresponds to the interdivision time ($\tau$).
    *   *Confidence*: High

*   **13-14: Kernel Trace: Septum Displacement / Length Normalized**
    *   *Description*: Arrays measuring the error in division site placement. The normalized version divides the displacement by the cell length, corresponding to $L_s$ in the paper.
    *   *Confidence*: High

*   **15-18: Kernel Trace: area / Length / Width / Volume**
    *   *Description*: Arrays containing the average size of cells (likely averaged over the entire cell cycle) at each experiment timepoint.
    *   *Confidence*: High

*   **19: Kernel Trace: mCherry mean_intensity**
    *   *Description*: Array of fluorescence intensity values. Based on the paper, this corresponds to the HU-mCherry fusion protein used to visualize the nucleoid (chromosome), quantifying DNA content or compaction.
    *   *Confidence*: High

*   **20-21: Kernel Trace: Instantaneous Growth Rate: Volume / Length**
    *   *Description*: Arrays of the single-cell growth rate ($\lambda$), calculated from volume or length extension.
    *   *Confidence*: High

*   **22-27: Transformed: z score**
    *   *Description*: Standardized versions (Z-scores) of the corresponding phenotypic traces (e.g., `Delta time`, `Length`, `Growth Rate`). These normalize the data relative to the library distribution, as mentioned in the paper for clustering analysis.
    *   *Confidence*: High

### **Identifiers & Metadata**
These columns provide identification for the genetic perturbation and the specific data source.

*   **28: Global CellID**
    *   *Description*: A unique identifier for a single cell. Since this dataframe aggregates data by sgRNA, this is likely the ID of a representative cell or lineage used for tracking or visualization.
    *   *Confidence*: Medium

*   **29: File Parquet Index**
    *   *Description*: Index used for data management within the Parquet file storage system.
    *   *Confidence*: High

*   **30-32: fov, row, trench**
    *   *Description*: Location coordinates of the microfluidic channel ("trench") where the cells were imaged. `fov` = Field of View.
    *   *Confidence*: High

*   **33: initial timepoints**
    *   *Description*: The starting timepoint (frame) of the trace or experiment.
    *   *Confidence*: Medium

*   **34-35: File Index, File Trench Index**
    *   *Description*: Internal file handling indices.
    *   *Confidence*: High

*   **36: CellID**
    *   *Description*: An integer identifier for the cell. Given the low number of unique values (~15), this likely represents the **generation number** or the index of the cell within its lineage tree.
    *   *Confidence*: Medium

*   **37: Trench Score**
    *   *Description*: A quality metric for the microfluidic trench (e.g., tracking stability).
    *   *Confidence*: High

*   **38-41: Mother / Daughter / Sister CellID**
    *   *Description*: Graph identifiers linking the representative cell to its relatives in the lineage tree.
    *   *Confidence*: High

*   **42-43: Centroid X / Y**
    *   *Description*: Spatial coordinates of the cell within the image.
    *   *Confidence*: High

*   **44-46: Kymograph / FOV Parquet Index**
    *   *Description*: Indices related to the kymograph (space-time image) generation and storage.
    *   *Confidence*: High

*   **47: Experiment #**
    *   *Description*: Identifier for the experimental replicate.
    *   *Confidence*: High

*   **48: phenotype trenchid**
    *   *Description*: Unique ID for the specific trench phenotype trace.
    *   *Confidence*: High

### **Genetics & Target**
*   **49: Barcode**
    *   *Description*: The unique nucleotide barcode sequence identifying the specific clone/lineage.
    *   *Confidence*: High

*   **50: sgRNA**
    *   *Description*: The sequence of the single guide RNA targeting the essential gene. This is the primary key for the rows in this dataframe.
    *   *Confidence*: High

*   **51: Closest Hamming Distance**
    *   *Description*: A sequencing quality metric indicating the distance to the nearest known barcode (used for error correction).
    *   *Confidence*: High

*   **52: EcoWG1_id**
    *   *Description*: An external database identifier for the E. coli gene (likely from the EcoCyc or similar database).
    *   *Confidence*: Medium

*   **53: Gene**
    *   *Description*: The common name of the targeted essential gene (e.g., "dnaA", "ftsZ").
    *   *Confidence*: High

*   **54: N Mismatch**
    *   *Description*: The number of mismatches designed into the sgRNA to modulate knockdown strength.
    *   *Confidence*: High

*   **55: Category**
    *   *Description*: Functional classification of the gene (e.g., "Essential", "Control").
    *   *Confidence*: High

*   **56: TargetID**
    *   *Description*: Numerical identifier for the gene target.
    *   *Confidence*: High

*   **57: barcodeid**
    *   *Description*: Numerical identifier for the barcode.
    *   *Confidence*: High

*   **58: N Observations**
    *   *Description*: The number of cells or lineages aggregated to compute the Kernel Traces for this sgRNA.
    *   *Confidence*: High

*   **59: Feature Vector**
    *   *Description*: A combined vector (likely high-dimensional) encapsulating the full phenotypic phenotype, used for clustering or dimensionality reduction.
    *   *Confidence*: High

### **Statistics (Error Bars)**
*   **60-88: SEM: [Feature]**
    *   *Description*: Standard Error of the Mean (SEM) for each of the corresponding "Kernel Trace" columns. These arrays quantify the variability or confidence of the population-averaged phenotype at each timepoint.
    *   *Confidence*: High

In [ ]:
first_row = df.iloc[0]
trace_cols = [
    c
    for c in df.columns
    if isinstance(first_row[c], (list, np.ndarray)) and len(first_row[c]) == 20
]
embed_cols = [
    c for c in df.columns if isinstance(first_row[c], (list, np.ndarray)) and len(first_row[c]) == 6
]

df_traces = df.drop(columns=embed_cols).explode(trace_cols)
df_traces[trace_cols] = df_traces[trace_cols].astype(float)
df_traces["time_point"] = df_traces.groupby(level=0).cumcount()
df_traces = df_traces.reset_index()


df_traces["Gene"] = pd.Categorical(df_traces["Gene"], categories=genes_in_path, ordered=True)
df_traces["Gene_group"] = np.where(df_traces["Gene"].isin(group1), "lpxACD", "lpxBK")

In [ ]:
# Feature	Group 1 (lpxA/D) Prediction	Group 2 (lpxB/K) Prediction
# Nucleoid (mCherry)	Normal / Dispersed	Highly Condensed / Intense (Due to dps)
# Growth State	Slow / Sick	Arrested / Dormant (Due to rpoS)
# Mechanism	Surface Remodeling	Cytoplasmic Toxicity

In [ ]:
(
    gg.ggplot(
        df_traces, gg.aes(x="time_point", y="Kernel Trace: mCherry mean_intensity", color="Gene")
    )
    + gg.geom_point()
    + gg.theme(figure_size=(10, 5))
)

In [ ]:
(
    gg.ggplot(
        df_traces,
        gg.aes(x="time_point", y="Kernel Trace: mCherry mean_intensity", fill="Gene"),
    )
    + gg.geom_point(color="black", stroke=0.2, size=2)
    + gg.theme(figure_size=(10, 5))
)

In [ ]:
(
    gg.ggplot(
        df_traces,
        gg.aes(x="time_point", y="Kernel Trace: Delta time (s)", fill="Gene"),
    )
    + gg.geom_point(color="black", stroke=0.2, size=2)
    + gg.theme(figure_size=(10, 5))
)

In [ ]:
def expand_embeddings(df, columns):
    expanded_dfs = []
    for col in columns:
        if col not in df.columns:
            continue
        col_values = np.stack(df[col].values)
        if col_values.ndim > 2:
            col_values = col_values.reshape(len(df), -1)

        expanded = pd.DataFrame(
            col_values, index=df.index, columns=[f"{col}_{i}" for i in range(col_values.shape[1])]
        )
        expanded_dfs.append(expanded)

    metadata_cols = [c for c in df.columns if c not in columns]
    return pd.concat([df[metadata_cols]] + expanded_dfs, axis=1).reset_index()


df_embeddings = expand_embeddings(timeseries_df, ["Feature Vector"])
df_embeddings.info()

In [ ]:
feature_vectors = df_embeddings[
    [col for col in df_embeddings.columns if col.startswith("Feature Vector")]
].fillna(0.0)

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
feature_vectors.values.dtype

In [ ]:
tsne_ = TSNE(n_components=2)
tsne_result = tsne_.fit_transform(feature_vectors.values)
df_embeddings["tsne_x"] = tsne_result[:, 0]
df_embeddings["tsne_y"] = tsne_result[:, 1]
df_embeddings

In [ ]:
df_embeddings_subset = df_embeddings[df_embeddings["Gene"].isin(group1 + group2)].loc[
    lambda x: x["N Mismatch"] == 0
]
df_embeddings_subset["Gene"] = pd.Categorical(
    df_embeddings_subset["Gene"], categories=genes_in_path, ordered=True
)

In [ ]:
(
    gg.ggplot(
        df_embeddings,
        gg.aes(x="tsne_x", y="tsne_y"),
    )
    + gg.geom_point()
    + gg.geom_point(df_embeddings_subset, gg.aes(x="tsne_x", y="tsne_y", color="Gene"))
    + gg.theme(figure_size=(10, 5))
)